# XGBoost

In [1]:
import os
import json
import pandas as pd
import numpy as np
from xgboost import XGBClassifier, XGBRanker
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
base_dir = os.path.dirname(os.path.abspath(__file__))
data_dir = os.path.join(base_dir, "..", "data", "split")

In [ ]:
ruta_train = os.path.join(data_dir, "train_split.csv")
ruta_test = os.path.join(data_dir, "test_split.csv")
ruta_val = os.path.join(data_dir, "val_split.csv")

ruta_metadata = os.path.join("game_recommendations_on_steam", "games_metadata.json")

In [153]:
def transformar_descripcion_a_str(dataset):
    dataset["descripciones"] = (dataset["descripciones"].fillna("").astype(str))
    return dataset

In [154]:
train_set = pd.read_csv(ruta_train)
test_set = pd.read_csv(ruta_test)

train_set["hours"] = np.log1p(train_set["hours"])
test_set["hours"] = np.log1p(test_set["hours"])

In [155]:
# Cargamos las descripciones
final_dict = {}

with open(ruta_metadata, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        app_id = obj["app_id"]
        final_dict[app_id] = str(obj["description"])
        

In [156]:
descripciones = list(final_dict.values())
keys_app_id = list(final_dict.keys())

In [157]:
train_set = train_set.sort_values("user_id").reset_index(drop=True)
test_set  = test_set.sort_values("user_id").reset_index(drop=True)

In [159]:
vectorizer = TfidfVectorizer(stop_words="english")
descripciones_train_tfid = vectorizer.fit_transform(descripciones)
dict_transformados = {i: j for i, j in zip(keys_app_id, descripciones_train_tfid)}
train_set["descripciones"] = train_set["app_id"].map(dict_transformados)


In [161]:
#descripciones_test_tfid = vectorizer.transform(test_set["descripciones"])
#dict_transformados_testset = {i: j for i, j in zip(keys_app_id, descripciones_test_tfid)}
test_set["descripciones"] = test_set["app_id"].map(dict_transformados)

In [ ]:
#train_set = transformar_descripcion_a_str(train_set)
#test_set = transformar_descripcion_a_str(test_set)

In [162]:
columnas_importantes = ["user_id", "app_id", "hours", "descripciones"] # app_id es el id del juego
train_set = train_set[columnas_importantes]
test_set = test_set[columnas_importantes]

In [ ]:
#columnas_train, columnas_predict = ["user_id", "app_id", "descripciones"], ["hours"]
#X_train, y_train = train_set[columnas_train], train_set[columnas_predict]
#X_test, y_test = test_set[columnas_train], test_set[columnas_predict]

In [166]:
import scipy.sparse

columnas_train, columnas_predict = ["user_id", "app_id", "descripciones"], ["hours"]
X_train_df, y_train = train_set[columnas_train], train_set[columnas_predict]
X_test_df, y_test = test_set[columnas_train], test_set[columnas_predict]

# Extract numerical features and convert to numpy arrays
numerical_features_train = X_train_df[["user_id", "app_id"]].values
numerical_features_test = X_test_df[["user_id", "app_id"]].values

# Convert the Series of sparse matrices in 'descripciones' column into a single sparse matrix
descripciones_sparse_train = scipy.sparse.vstack(X_train_df["descripciones"].tolist())
descripciones_sparse_test = scipy.sparse.vstack(X_test_df["descripciones"].tolist())

# Horizontally stack numerical features and sparse features
X_train = scipy.sparse.hstack((numerical_features_train, descripciones_sparse_train))
X_test = scipy.sparse.hstack((numerical_features_test, descripciones_sparse_test))

In [ ]:
group_train = train_set.groupby("user_id").size().tolist()
group_test  = test_set.groupby("user_id").size().tolist()

In [187]:
# Esto hay que tunearlo
ranker = XGBRanker(
    objective="rank:pairwise",
    learning_rate=0.1,
    n_estimators=300,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

ranker.fit(
    X_train,
    y_train,
    group=group_train
)


,objective,'rank:pairwise'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
y_pred = ranker.predict(X_test)

In [189]:
print("Iterating through each row of X_test and showing predicted scores along with actual hours:")
# You can change the range to iterate through all rows if needed.
# For demonstration, let's show the first 10 rows.
for i in range(X_test_df.shape[0]):
    user_id = X_test_df.iloc[i]['user_id']
    app_id = X_test_df.iloc[i]['app_id']
    predicted_score = y_pred[i]
    actual_hours = y_test.iloc[i]['hours'] # Get the actual hours from y_test
    print(f"Row {i}: User ID: {user_id}, App ID: {app_id}, Predicted Score: {predicted_score:.4f}, Actual Hours: {actual_hours:.4f}")

Iterating through each row of X_test and showing predicted scores along with actual hours:
Row 0: User ID: 731, App ID: 42960, Predicted Score: -0.0855, Actual Hours: 0.2624
Row 1: User ID: 3128, App ID: 368260, Predicted Score: 0.1459, Actual Hours: 2.3514
Row 2: User ID: 4232, App ID: 33230, Predicted Score: -0.0367, Actual Hours: 1.9741
Row 3: User ID: 8297, App ID: 1938090, Predicted Score: -0.0600, Actual Hours: 4.3870
Row 4: User ID: 9905, App ID: 1377580, Predicted Score: -0.1444, Actual Hours: 5.6366
Row 5: User ID: 10199, App ID: 1361210, Predicted Score: 0.0080, Actual Hours: 2.1861
Row 6: User ID: 13360, App ID: 1938090, Predicted Score: -0.0600, Actual Hours: 5.8108
Row 7: User ID: 14077, App ID: 1583320, Predicted Score: -0.0334, Actual Hours: 2.5649
Row 8: User ID: 16342, App ID: 1229490, Predicted Score: 0.0038, Actual Hours: 2.8959
Row 9: User ID: 18612, App ID: 227940, Predicted Score: -0.1565, Actual Hours: 4.2739
Row 10: User ID: 20886, App ID: 1240440, Predicted Sco